# Falcon Challenge - descargas de datos

Notebook de apoyo para los puntos 8 y 9 de `FalconChallenge_V6.pdf`. Centraliza las fuentes IBWC/USIBWC, la carpeta compartida del benchmark y ejemplos para descargar o guardar los recursos relacionados con Falcon Reservoir.

## Fuentes del challenge

Punto 8, disponibilidad de datos:

- Portal IBWC Water Data: `https://www.ibwc.gov/water-data/`
- Dashboard AQWebportal: `https://waterdata.ibwc.gov/AQWebportal/Data/Dashboard/5`
- Estacion `08461200`: International Falcon Reservoir.
- Estacion `08461300`: Rio Grande Below Falcon Dam.

Punto 9, datos opcionales/escalado:

- Carpeta compartida del benchmark en SharePoint.
- Reportes USIBWC de condiciones diarias, almacenamiento, ownership semanal y reportes de Falcon.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
from html.parser import HTMLParser
import json
import mimetypes
import re

import pandas as pd
import requests

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

http = requests.Session()
http.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,es;q=0.8',
    'Referer': 'https://www.ibwc.gov/',
})

pd.set_option('display.max_colwidth', 140)


In [ ]:
source_urls = {
    "ibwc_water_data": "https://www.ibwc.gov/water-data/",
    "aqwebportal_dashboard": "https://waterdata.ibwc.gov/AQWebportal/Data/Dashboard/5",
    "sharepoint_benchmark_folder": (
        "https://cicesemx0-my.sharepoint.com/:f:/g/personal/fadomin_cicese_mx/"
        "IgDamhhYBeWfSb73hdPj-EmXAWlRxnqXUyBlI0u9GUiJvrM?e=YDLUL0"
    ),
    "daily_rio_grande_flow_conditions": "https://ibwcsftpstg.blob.core.windows.net/wad/DailyReports/flowdata.htm",
    "weekly_ownership_reports": "https://ibwcsftpstg.blob.core.windows.net/wad/WeeklyReports/storage.htm",
    "falcon_reservoir_report_image": "https://ibwcsftpstg.blob.core.windows.net/wad/WeeklyReports/falsto.gif",
    "amistad_falcon_ownership_trends_image": "https://ibwcsftpstg.blob.core.windows.net/wad/WeeklyReports/amfalpct.gif",
}

ibwc_station_datasets = [
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Total Storage.Web-Daily-tcm@08461200",
        "use": "storage time series",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Total Storage.Web-Telemetry-tcm@08461200",
        "use": "near-real-time storage",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Reservoir Elevation.Web-Daily-m@08461200",
        "use": "reservoir elevation",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Lake Area.Best Available@08461200",
        "use": "lake area",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Evaporation,accumltd.Daily Evaporation - mm@08461200",
        "use": "evaporation",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Discharge.Total.Change-in-Storage@08461200",
        "use": "storage change / inferred balance",
    },
    {
        "station": "08461200",
        "station_name": "International Falcon Reservoir",
        "dataset": "Percentage.Conservation-Web-Telemetry@08461200",
        "use": "conservation percentage",
    },
    {
        "station": "08461300",
        "station_name": "Rio Grande Below Falcon Dam",
        "dataset": "Discharge.Best Available@08461300",
        "use": "observed historical release from Falcon Dam",
    },
]

pd.DataFrame(ibwc_station_datasets)

fallback_ibwc_links = [
    {
        'text': 'Daily Rio Grande Flow Conditions',
        'url': source_urls['daily_rio_grande_flow_conditions'],
        'source': 'IBWC Water Data page',
    },
    {
        'text': 'Weekly Ownership Reports',
        'url': source_urls['weekly_ownership_reports'],
        'source': 'IBWC Water Data page',
    },
    {
        'text': 'Falcon Reservoir Report',
        'url': source_urls['falcon_reservoir_report_image'],
        'source': 'IBWC Water Data page',
    },
    {
        'text': 'Amistad-Falcon Ownership Trends 1996-present',
        'url': source_urls['amistad_falcon_ownership_trends_image'],
        'source': 'IBWC Water Data page',
    },
    {
        'text': 'AQWebportal Dashboard 5',
        'url': source_urls['aqwebportal_dashboard'],
        'source': 'Falcon Challenge point 8',
    },
]

pd.DataFrame(fallback_ibwc_links)


## Utilidades de descarga

Estas funciones guardan recursos directos y listan enlaces de paginas HTML sin depender de `beautifulsoup4`, `lxml` ni `openpyxl`.

In [ ]:
def safe_filename_from_url(url: str, fallback: str = 'download.bin') -> str:
    name = url.rstrip('/').split('/')[-1].split('?')[0]
    return name or fallback


def download_file(url: str, output_path: Path | None = None, timeout: int = 120) -> Path:
    if output_path is None:
        output_path = RAW_DIR / safe_filename_from_url(url)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with http.get(url, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        with output_path.open('wb') as fh:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fh.write(chunk)
    return output_path


def try_download_file(url: str, output_path: Path | None = None, timeout: int = 120):
    try:
        return {'ok': True, 'path': download_file(url, output_path, timeout=timeout), 'error': None}
    except Exception as exc:
        return {'ok': False, 'path': output_path, 'error': f'{type(exc).__name__}: {exc}'}


class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self._href = None
        self._text = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == 'a':
            attrs = dict(attrs)
            self._href = attrs.get('href')
            self._text = []

    def handle_data(self, data):
        if self._href:
            self._text.append(data)

    def handle_endtag(self, tag):
        if tag.lower() == 'a' and self._href:
            self.links.append({'text': ' '.join(self._text).strip(), 'href': self._href})
            self._href = None
            self._text = []


def find_links(page_url: str) -> pd.DataFrame:
    response = http.get(page_url, timeout=60)
    response.raise_for_status()
    parser = LinkParser()
    parser.feed(response.text)
    rows = [
        {'text': item['text'], 'url': urljoin(page_url, item['href'])}
        for item in parser.links
    ]
    return pd.DataFrame(rows).drop_duplicates('url') if rows else pd.DataFrame(columns=['text', 'url'])


def try_find_links(page_url: str) -> pd.DataFrame:
    try:
        return find_links(page_url)
    except Exception as exc:
        print(f'No se pudieron listar enlaces de {page_url}: {type(exc).__name__}: {exc}')
        return pd.DataFrame(columns=['text', 'url'])


## 1. Listar enlaces del portal IBWC Water Data

El punto 9 recomienda empezar por `Daily Rio Grande Flow Conditions` y enlaces de reservorios/ownership. Esta celda genera un indice local de enlaces publicados en la pagina oficial.

In [ ]:
ibwc_links = try_find_links(source_urls['ibwc_water_data'])

if ibwc_links.empty:
    print('El sitio IBWC devolvio 403 o no expuso enlaces al scraper. Uso fallback con enlaces oficiales conocidos.')
    ibwc_links = pd.DataFrame(fallback_ibwc_links)
else:
    ibwc_links['source'] = 'scraped from IBWC Water Data page'

ibwc_links.to_csv(DATA_DIR / 'ibwc_water_data_links.csv', index=False)
mask = ibwc_links['url'].str.contains('DailyReports|WeeklyReports|AQWebportal|Falcon|flow|storage|falsto|amfalpct', case=False, regex=True, na=False)
display(ibwc_links[mask])


## 2. Descargar reportes directos de USIBWC

Estos recursos son enlaces directos desde la pagina de Water Data: condiciones diarias, ownership semanal, imagen del reporte Falcon y tendencia Amistad-Falcon.

In [ ]:
direct_downloads = {
    'daily_rio_grande_flow_conditions': RAW_DIR / 'ibwc_daily_rio_grande_flow_conditions.htm',
    'weekly_ownership_reports': RAW_DIR / 'ibwc_weekly_ownership_reports.htm',
    'falcon_reservoir_report_image': RAW_DIR / 'falcon_reservoir_report.gif',
    'amistad_falcon_ownership_trends_image': RAW_DIR / 'amistad_falcon_ownership_trends.gif',
}

download_results = []
for key, output_path in direct_downloads.items():
    result = try_download_file(source_urls[key], output_path)
    download_results.append({'dataset': key, 'url': source_urls[key], **result})

download_results_df = pd.DataFrame(download_results)
display(download_results_df)


## 3. Extraer texto util del reporte semanal

La pagina semanal incluye almacenamiento, releases, inflows y losses de Amistad/Falcon. Esta celda guarda una version de texto plano y muestra el bloque alrededor de `Falcon Dam`.

In [ ]:
weekly_path = RAW_DIR / 'ibwc_weekly_ownership_reports.htm'

if not weekly_path.exists():
    print('Primero descarga el reporte semanal o guarda manualmente el HTML en:', weekly_path)
else:
    weekly_html = weekly_path.read_text(encoding='utf-8', errors='ignore')
    weekly_text = re.sub(r'<[^>]+>', ' ', weekly_html)
    weekly_text = re.sub(r'\s+', ' ', weekly_text).strip()
    (DATA_DIR / 'ibwc_weekly_ownership_reports.txt').write_text(weekly_text, encoding='utf-8')

    match = re.search(r'Falcon Dam(.+?)(?:Note:|Last Updated|$)', weekly_text, flags=re.IGNORECASE)
    falcon_weekly_text = match.group(0) if match else weekly_text
    print(falcon_weekly_text[:2500])


## 4. Dashboard AQWebportal

El dashboard es la fuente indicada para las series `08461200` y `08461300`. Esta celda guarda la pagina y busca rutas que parezcan endpoints de datos, excluyendo assets estaticos como favicon, logos, CSS, imagenes y fuentes. Si no encuentra endpoints utiles, muestra los datasets que hay que exportar desde la interfaz.


In [ ]:
dashboard_result = try_download_file(source_urls['aqwebportal_dashboard'], RAW_DIR / 'aqwebportal_dashboard_5.html')
print(dashboard_result)

expected_dashboard_exports = pd.DataFrame(ibwc_station_datasets)[['station', 'station_name', 'dataset', 'use']]

if dashboard_result['ok']:
    dashboard_html = Path(dashboard_result['path']).read_text(encoding='utf-8', errors='ignore')

    raw_urls = sorted(set(re.findall(r'https?://[^\"\'<>\s)]+|/[A-Za-z0-9_./?=&%:-]+', dashboard_html)))
    candidate_urls = [urljoin(source_urls['aqwebportal_dashboard'], u) for u in raw_urls]

    static_asset_pattern = re.compile(
        r'\.(?:ico|png|jpg|jpeg|gif|svg|webp|css|woff|woff2|ttf|eot|map)(?:\?|$)|favicon|logo|bootstrap|jquery|fontawesome',
        flags=re.IGNORECASE,
    )
    data_endpoint_pattern = re.compile(
        r'(?:api|export|csv|json|time.?series|timeseries|download|datatable|data/|Get[A-Z]|Chart|Station|Parameter|Dashboard)',
        flags=re.IGNORECASE,
    )

    api_candidates = [
        u for u in candidate_urls
        if data_endpoint_pattern.search(u) and not static_asset_pattern.search(u)
    ]

    api_candidates_df = pd.DataFrame({'url': sorted(set(api_candidates))})
    api_candidates_df.to_csv(DATA_DIR / 'aqwebportal_candidate_endpoints.csv', index=False)

    if api_candidates_df.empty:
        print('No se encontraron endpoints de datos en el HTML inicial. Exporta estos datasets desde el dashboard:')
        display(expected_dashboard_exports)
    else:
        display(api_candidates_df.head(50))
        print('Si la tabla solo contiene rutas internas no descargables, exporta manualmente los datasets esperados:')
        display(expected_dashboard_exports)
else:
    print('Abre manualmente el dashboard y exporta las series necesarias:', source_urls['aqwebportal_dashboard'])
    display(expected_dashboard_exports)


## 5. Guardar exportaciones manuales del AQWebportal

Si el dashboard requiere seleccion manual, abre `source_urls["aqwebportal_dashboard"]`, selecciona la estacion y dataset del cuadro anterior, usa la opcion de exportar/descargar y pega aqui la URL final del archivo generado.

In [ ]:
# Pega aqui una URL directa generada por el dashboard, por ejemplo un CSV o JSON.
aqwebportal_export_url = ''

if aqwebportal_export_url:
    try:
        content_type = http.head(aqwebportal_export_url, timeout=30).headers.get('content-type', '')
        suffix = mimetypes.guess_extension(content_type) or '.dat'
    except Exception:
        suffix = '.dat'
    output = RAW_DIR / f'aqwebportal_export{suffix}'
    print(try_download_file(aqwebportal_export_url, output))
else:
    print('Pendiente: pega una URL de exportacion del dashboard AQWebportal.')


## 6. Carpeta compartida del benchmark

El punto 9 indica que el dataset oficial completo se proporciona en una carpeta compartida. Si el enlace requiere sesion o confirmacion del navegador, descargalo desde la interfaz de SharePoint y coloca los archivos en `data/raw/sharepoint_benchmark/`. Si obtienes un enlace directo de descarga, pegalo en `sharepoint_direct_download_url`.

In [ ]:
sharepoint_dir = RAW_DIR / "sharepoint_benchmark"
sharepoint_dir.mkdir(parents=True, exist_ok=True)

print("Carpeta benchmark:", source_urls["sharepoint_benchmark_folder"])
print("Destino local sugerido:", sharepoint_dir)

sharepoint_direct_download_url = ""
if sharepoint_direct_download_url:
    print(download_file(sharepoint_direct_download_url, sharepoint_dir / safe_filename_from_url(sharepoint_direct_download_url)))

## 7. Manifest local

Guarda un manifiesto reproducible de fuentes, estaciones y datasets esperados. Sirve para documentar que las series oficiales son Falcon storage (`08461200`) y observed release (`08461300`).

In [ ]:
manifest = {
    "source_urls": source_urls,
    "ibwc_station_datasets": ibwc_station_datasets,
    "notes": [
        "Use Discharge.Best Available@08461300 as observed historical release.",
        "If discharge is in m3/s, convert it to volume over the selected time step before storage-balance modeling.",
        "Official benchmark files from SharePoint should not be replaced by optional scaling data.",
    ],
}

manifest_path = DATA_DIR / "falcon_download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest_path